# Load Aircraft Graph into Neo4j

Loads the full aircraft digital twin dataset into Neo4j from CSV files in a UC Volume.

**Prerequisites:**
- Run `sample-validation/upload_data.sh` to upload CSV files to the UC Volume
- Run `sample-validation/create_secrets.sh` to store Neo4j credentials as Databricks secrets

**Graph loaded:**
- 20 Aircraft, 12 Airports, 80 Systems, 320 Components, 160 Sensors
- 800 Flights, 300 Maintenance Events, 514 Delays
- 8 relationship types connecting all node labels

Run this notebook once before notebooks 01, 02, and 03.

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these values for your environment
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://<instance>.databases.neo4j.io"
SECRET_SCOPE = "sample_validation"
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "aircraft_data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL_SQL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## CSV Helper

Reads Neo4j import-format CSVs and normalizes column headers.

In [ ]:
import csv

def csv_rows(path):
    """Read a Neo4j import CSV and normalize column names.

    Converts Neo4j import column prefixes to plain names:
      :ID(Label)       → id
      :START_ID(Label) → start_id
      :END_ID(Label)   → end_id
    """
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            norm = {}
            for k, v in row.items():
                if k.startswith(":ID("):
                    norm["id"] = v
                elif k.startswith(":START_ID("):
                    norm["start_id"] = v
                elif k.startswith(":END_ID("):
                    norm["end_id"] = v
                else:
                    norm[k] = v
            rows.append(norm)
    return rows

---

## Section 1: Clear Existing Data

Removes all nodes and relationships so the graph loads cleanly.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

print("--- Section 1: Clear Existing Data ---")
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("[PASS] Cleared existing data")

---

## Section 2: Create Indexes

Creates indexes on the primary ID property for each node label.

In [ ]:
print("--- Section 2: Create Indexes ---")
indexes = [
    "CREATE INDEX aircraft_id IF NOT EXISTS FOR (n:Aircraft) ON (n.aircraftId)",
    "CREATE INDEX airport_id  IF NOT EXISTS FOR (n:Airport)  ON (n.airportId)",
    "CREATE INDEX system_id   IF NOT EXISTS FOR (n:System)   ON (n.systemId)",
    "CREATE INDEX component_id IF NOT EXISTS FOR (n:Component) ON (n.componentId)",
    "CREATE INDEX sensor_id   IF NOT EXISTS FOR (n:Sensor)   ON (n.sensorId)",
    "CREATE INDEX flight_id   IF NOT EXISTS FOR (n:Flight)   ON (n.flightId)",
    "CREATE INDEX delay_id    IF NOT EXISTS FOR (n:Delay)    ON (n.delayId)",
    "CREATE INDEX maint_id    IF NOT EXISTS FOR (n:MaintenanceEvent) ON (n.eventId)",
]
with driver.session() as session:
    for q in indexes:
        session.run(q)
print(f"[PASS] {len(indexes)} indexes created")

---

## Section 3: Load Node Types

Loads each node label from its CSV file in the UC Volume.

In [ ]:
print("--- Section 3: Load Nodes ---")
nodes = [
    ("Aircraft", f"{VOLUME_PATH}/nodes_aircraft.csv", """
        UNWIND $rows AS row
        CREATE (:Aircraft {aircraftId: row.id, tail_number: row.tail_number,
                           icao24: row.icao24, model: row.model,
                           manufacturer: row.manufacturer, operator: row.operator})
    """, 20),
    ("Airport", f"{VOLUME_PATH}/nodes_airports.csv", """
        UNWIND $rows AS row
        CREATE (:Airport {airportId: row.id, name: row.name, city: row.city,
                          country: row.country, iata: row.iata, icao: row.icao,
                          lat: toFloat(row.lat), lon: toFloat(row.lon)})
    """, 12),
    ("System", f"{VOLUME_PATH}/nodes_systems.csv", """
        UNWIND $rows AS row
        CREATE (:System {systemId: row.id, aircraftId: row.aircraft_id,
                         type: row.type, name: row.name})
    """, 80),
    ("Component", f"{VOLUME_PATH}/nodes_components.csv", """
        UNWIND $rows AS row
        CREATE (:Component {componentId: row.id, systemId: row.system_id,
                            type: row.type, name: row.name})
    """, 320),
    ("Sensor", f"{VOLUME_PATH}/nodes_sensors.csv", """
        UNWIND $rows AS row
        CREATE (:Sensor {sensorId: row.id, systemId: row.system_id,
                         type: row.type, name: row.name, unit: row.unit})
    """, 160),
    ("Flight", f"{VOLUME_PATH}/nodes_flights.csv", """
        UNWIND $rows AS row
        CREATE (:Flight {flightId: row.id, flight_number: row.flight_number,
                         aircraftId: row.aircraft_id, operator: row.operator,
                         origin: row.origin, destination: row.destination,
                         scheduled_departure: row.scheduled_departure,
                         scheduled_arrival: row.scheduled_arrival})
    """, 800),
    ("MaintenanceEvent", f"{VOLUME_PATH}/nodes_maintenance.csv", """
        UNWIND $rows AS row
        CREATE (:MaintenanceEvent {eventId: row.id, componentId: row.component_id,
                                   systemId: row.system_id, aircraftId: row.aircraft_id,
                                   fault: row.fault, severity: row.severity,
                                   reported_at: row.reported_at,
                                   corrective_action: row.corrective_action})
    """, 300),
    ("Delay", f"{VOLUME_PATH}/nodes_delays.csv", """
        UNWIND $rows AS row
        CREATE (:Delay {delayId: row.id, flightId: row.flight_id,
                        cause: row.cause, minutes: toInteger(row.minutes)})
    """, 514),
]

with driver.session() as session:
    for label, csv_path, query, expected in nodes:
        session.run(query, rows=csv_rows(csv_path))
        count = session.run(f"MATCH (n:{label}) RETURN count(n) AS cnt").single()["cnt"]
        status = "PASS" if count == expected else "FAIL"
        print(f"  [{status}] {label}: {count} nodes")

---

## Section 4: Load Relationships

Creates all edges between nodes using the relationship CSV files.

In [ ]:
print("--- Section 4: Load Relationships ---")
relationships = [
    ("HAS_SYSTEM",      f"{VOLUME_PATH}/rels_aircraft_system.csv",
     "MATCH (a:Aircraft {aircraftId: row.start_id}) "
     "MATCH (s:System   {systemId:   row.end_id})   CREATE (a)-[:HAS_SYSTEM]->(s)"),
    ("HAS_COMPONENT",   f"{VOLUME_PATH}/rels_system_component.csv",
     "MATCH (s:System    {systemId:    row.start_id}) "
     "MATCH (c:Component {componentId: row.end_id})  CREATE (s)-[:HAS_COMPONENT]->(c)"),
    ("HAS_SENSOR",      f"{VOLUME_PATH}/rels_system_sensor.csv",
     "MATCH (s:System {systemId: row.start_id}) "
     "MATCH (sn:Sensor {sensorId: row.end_id}) CREATE (s)-[:HAS_SENSOR]->(sn)"),
    ("OPERATES_FLIGHT", f"{VOLUME_PATH}/rels_aircraft_flight.csv",
     "MATCH (a:Aircraft {aircraftId: row.start_id}) "
     "MATCH (f:Flight   {flightId:   row.end_id})   CREATE (a)-[:OPERATES_FLIGHT]->(f)"),
    ("DEPARTS_FROM",    f"{VOLUME_PATH}/rels_flight_departure.csv",
     "MATCH (f:Flight  {flightId:  row.start_id}) "
     "MATCH (a:Airport {airportId: row.end_id})   CREATE (f)-[:DEPARTS_FROM]->(a)"),
    ("ARRIVES_AT",      f"{VOLUME_PATH}/rels_flight_arrival.csv",
     "MATCH (f:Flight  {flightId:  row.start_id}) "
     "MATCH (a:Airport {airportId: row.end_id})   CREATE (f)-[:ARRIVES_AT]->(a)"),
    ("HAS_DELAY",       f"{VOLUME_PATH}/rels_flight_delay.csv",
     "MATCH (f:Flight {flightId: row.start_id}) "
     "MATCH (d:Delay  {delayId:  row.end_id})   CREATE (f)-[:HAS_DELAY]->(d)"),
    ("HAS_EVENT",       f"{VOLUME_PATH}/rels_component_event.csv",
     "MATCH (c:Component       {componentId: row.start_id}) "
     "MATCH (m:MaintenanceEvent {eventId:    row.end_id})   CREATE (c)-[:HAS_EVENT]->(m)"),
]

with driver.session() as session:
    for rel_type, csv_path, match_create in relationships:
        session.run(f"UNWIND $rows AS row {match_create}", rows=csv_rows(csv_path))
        print(f"  [PASS] {rel_type}")

---

## Section 5: Verify Counts

Confirms all expected nodes loaded correctly.

In [ ]:
print("--- Section 5: Verify Counts ---")
expected_counts = {
    "Aircraft": 20, "Airport": 12, "System": 80, "Component": 320,
    "Sensor": 160, "Flight": 800, "MaintenanceEvent": 300, "Delay": 514,
}
with driver.session() as session:
    for label, expected in expected_counts.items():
        count = session.run(f"MATCH (n:{label}) RETURN count(n) AS cnt").single()["cnt"]
        status = "PASS" if count == expected else "FAIL"
        print(f"  [{status}] {label}: {count} nodes")
driver.close()
print("\nGraph load complete. Run 01-simple-connect-test.ipynb next.")